<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Gemma3_Multimodal.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

### Gemma 3 Multimodal Demo with TransformerBridge

This notebook demonstrates how to use TransformerBridge with `Gemma3ForConditionalGeneration`,
the vision-language variant of Gemma 3. The model pairs a SigLIP vision encoder with the
Gemma 3 language model and is the same architecture used by MedGemma.

We demonstrate:
1. Loading Gemma 3 (4B-it) through TransformerBridge
2. Multimodal generation from an image + text prompt
3. Capturing vision-language activations with `run_with_cache()`

> **Gated model.** The `google/gemma-3-*` checkpoints are gated on Hugging Face. Accept
> the license at https://huggingface.co/google/gemma-3-4b-it and run `huggingface-cli login`
> (or set `HF_TOKEN`) before executing this notebook.

In [ ]:
# Detect Colab and install dependencies if needed
DEVELOPMENT_MODE = False
try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    %pip install transformer_lens
    %pip install circuitsvis
except:
    IN_COLAB = False

In [ ]:
# NBVAL_IGNORE_OUTPUT
import torch
from PIL import Image
import requests
from io import BytesIO

device = 'cuda' if torch.cuda.is_available() else 'cpu'

import matplotlib.pyplot as plt
%matplotlib inline

from transformer_lens.model_bridge import TransformerBridge

try:
    import circuitsvis as cv
except ImportError:
    print('circuitsvis not installed, attention visualization will not work')
    cv = None

## Load Gemma 3 through TransformerBridge

TransformerBridge maps `Gemma3ForConditionalGeneration` to its multimodal adapter, which
wraps the SigLIP vision tower, the multimodal projector, and the Gemma 3 language model
into a single hooked model.

We use **bfloat16** here — Gemma 3 is trained in bf16 and fp16 can produce unstable activations.
The 4B-it variant is the smallest multimodal Gemma 3 (the 270m and 1B checkpoints are text-only).

In [ ]:
# NBVAL_IGNORE_OUTPUT
model = TransformerBridge.boot_transformers(
    "google/gemma-3-4b-it",
    device=device,
    dtype=torch.bfloat16,
)

for param in model.parameters():
    param.requires_grad = False

print(f"Model loaded on {device}")
print(f"Multimodal: {getattr(model.cfg, 'is_multimodal', False)}")
print(f"Layers: {model.cfg.n_layers}, Heads: {model.cfg.n_heads}")
print(f"Vision tokens per image: {getattr(model.cfg, 'mm_tokens_per_image', None)}")

## Load a test image

We'll use a stop-sign photo from Australia to test the model's visual understanding.

In [ ]:
# NBVAL_IGNORE_OUTPUT
image_url = "https://www.ilankelman.org/stopsigns/australia.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content)).convert("RGB")
plt.imshow(image)
plt.axis('off')
plt.title('Test Image')
plt.show()

## Multimodal Generation

Gemma 3 instruction-tuned models expect the chat template format with
`<start_of_turn>` / `<end_of_turn>` markers, and use `<start_of_image>` as the image
placeholder (rather than LLaVA's `<image>`). The processor expands `<start_of_image>`
into the appropriate number of vision tokens (256 by default for Gemma 3 4B).

We call `prepare_multimodal_inputs()` to run the processor on text + image, then pass
`pixel_values` to `generate()`. The bridge's `generate()` keeps a KV cache
(`use_past_kv_cache=True` by default) for efficient autoregressive decoding while
preserving full hook access.

In [ ]:
# NBVAL_IGNORE_OUTPUT
question = "What do you see in this photo?"
prompt = (
    "<start_of_turn>user\n"
    f"<start_of_image>{question}<end_of_turn>\n"
    "<start_of_turn>model\n"
)

# Prepare multimodal inputs (handles image processing + tokenization)
inputs = model.prepare_multimodal_inputs(text=prompt, images=image)
input_ids = inputs['input_ids']
pixel_values = inputs['pixel_values']

# Pass any extra processor outputs (e.g. token_type_ids for Gemma 3)
extra_kwargs = {k: v for k, v in inputs.items()
                if k not in ('input_ids', 'pixel_values')}

generated_text = model.generate(
    input_ids,
    pixel_values=pixel_values,
    max_new_tokens=80,
    do_sample=False,
    use_past_kv_cache=True,
    return_type="str",
    **extra_kwargs,
)

print('Generated text:', generated_text)

Let's try a second image to confirm the model adapts its description:

In [ ]:
# NBVAL_IGNORE_OUTPUT
image_url_2 = "https://github.com/zazamrykh/PicFinder/blob/main/images/doge.jpg?raw=true"
response = requests.get(image_url_2)
image_2 = Image.open(BytesIO(response.content)).convert("RGB")
plt.imshow(image_2)
plt.axis('off')
plt.show()

inputs = model.prepare_multimodal_inputs(text=prompt, images=image_2)
input_ids = inputs['input_ids']
pixel_values = inputs['pixel_values']
extra_kwargs = {k: v for k, v in inputs.items()
                if k not in ('input_ids', 'pixel_values')}

generated_text = model.generate(
    input_ids,
    pixel_values=pixel_values,
    max_new_tokens=80,
    do_sample=False,
    use_past_kv_cache=True,
    return_type="str",
    **extra_kwargs,
)
print('Generated text:', generated_text)

## Inspecting Vision-Language Activations

`run_with_cache()` accepts the same `pixel_values` argument and captures activations from
the vision encoder, the multimodal projector, and every transformer block in the language
model. This lets us inspect how the language tokens attend to image tokens during
multimodal processing.

In [ ]:
# NBVAL_IGNORE_OUTPUT
inputs = model.prepare_multimodal_inputs(text=prompt, images=image)
extra_kwargs = {k: v for k, v in inputs.items()
                if k not in ('input_ids', 'pixel_values')}

with torch.no_grad():
    logits, cache = model.run_with_cache(
        inputs['input_ids'],
        pixel_values=inputs['pixel_values'],
        **extra_kwargs,
    )

print(f'Logits shape: {logits.shape}')
print(f'Cache entries: {len(cache)}')
vision_keys = [k for k in cache.keys() if 'vision' in k.lower()]
print(f'Vision-related cache entries: {len(vision_keys)}')
print(f'Sample vision keys: {vision_keys[:5]}')

In [ ]:
# NBVAL_IGNORE_OUTPUT
if cv is not None:
    layer_to_visualize = 16
    tokens_to_show = 30

    pattern_keys = [k for k in cache.keys() if f'blocks.{layer_to_visualize}' in k and 'pattern' in k]
    if pattern_keys:
        attention_pattern = cache[pattern_keys[0]]
        if attention_pattern.ndim == 4:
            attention_pattern = attention_pattern[0]

        token_ids = inputs['input_ids'][0].cpu()
        str_tokens = model.tokenizer.convert_ids_to_tokens(token_ids)

        print(f'Layer {layer_to_visualize} Head Attention Patterns (last {tokens_to_show} tokens):')
        display(cv.attention.attention_patterns(
            tokens=str_tokens[-tokens_to_show:],
            attention=attention_pattern[:, -tokens_to_show:, -tokens_to_show:].float().cpu(),
        ))
    else:
        print(f'No attention pattern found for layer {layer_to_visualize}')
        print(f'Available attention-related keys: {[k for k in cache.keys() if "attn" in k][:10]}')
else:
    print('circuitsvis not available — skipping visualization')

## Summary

TransformerBridge provides native multimodal support for `Gemma3ForConditionalGeneration`:

- **`boot_transformers("google/gemma-3-4b-it")`** loads the full vision + projector + language pipeline
- **`prepare_multimodal_inputs(text=..., images=...)`** handles image processing and tokenization
- **`generate(input_ids, pixel_values=...)`** runs multimodal generation with KV cache and hooks
- **`run_with_cache(input_ids, pixel_values=...)`** captures activations including SigLIP vision tokens

A few Gemma 3 specifics worth noting:

- Use the chat-template format (`<start_of_turn>user ... <end_of_turn><start_of_turn>model`) for instruction-tuned variants
- The image placeholder is `<start_of_image>`, not `<image>`
- Gemma 3 is trained in bf16 — prefer `torch.bfloat16` over `torch.float16`
- The same code path works for MedGemma (`google/medgemma-4b-it`, `google/medgemma-27b-it`) and any other `Gemma3ForConditionalGeneration` checkpoint